# 02 · Baseline extraction on the golden set

Establishes the number the LLM must beat. Two engines:

- **heuristic** — regex baseline (free, runs here).
- **llm** — schema-guided extraction via the self-hosted vLLM. Run it with:
  `cd ai-service && uv run python ../evals/run_evals.py --engine llm --golden v0`

Both write results + W&B runs; this notebook loads and compares them.

In [1]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ai-service"))
sys.path.insert(0, str(ROOT / "evals"))
FIG = ROOT / "notebooks" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("repo root:", ROOT)


repo root: /Users/zaid/expenseiq


In [2]:
import json, sys
from run_evals import load_golden, run_heuristic, score
records = load_golden('v0')
print('golden v0 size:', len(records))

golden v0 size: 75


## Run the heuristic baseline

In [3]:
preds = run_heuristic(records)
summary = score(records, preds)
import pandas as pd
pd.Series(summary['field_accuracy']).to_frame('accuracy').join(
    pd.Series(summary['field_support']).to_frame('support'))

,accuracy,support
merchant,0.6400,50
total_amount,0.3467,75
transaction_date,0.8400,50


## Field accuracy vs exact-document match

In [4]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
acc = {k:(v or 0) for k,v in summary['field_accuracy'].items()}; acc['exact_doc']=summary['exact_doc_match']
ax.bar(list(acc), list(acc.values()), color='#4C72B0')
ax.set_ylim(0,1); ax.set_ylabel('accuracy'); ax.set_title('Heuristic baseline')
for i,(k,v) in enumerate(acc.items()): ax.text(i, v+0.02, f'{v:.2f}', ha='center')
fig.tight_layout(); fig.savefig(FIG/'heuristic_baseline.png', dpi=120); plt.show()
print('exact_doc_match =', summary['exact_doc_match'])

exact_doc_match = 0.1467


/var/folders/n0/vd3cdjmj6b5895m4qc0392d40000gn/T/ipykernel_5453/957898246.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIG/'heuristic_baseline.png', dpi=120); plt.show()


## LLM baseline

Once the GPU/vLLM endpoint is up, run the LLM engine and load `evals/results/llm-v0-*.json` here to compare accuracy, p95 latency and cost/doc. The self-healing harness records how many documents were auto-repaired.